# Employee Reviews — Data Cleaning & Preprocessing
This notebook cleans and preprocesses a Glassdoor-style employee reviews dataset.

In [27]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
import openpyxl

pd.set_option('display.max_columns', 100)
DATA_DIR = Path('.')
OUTPUT_DIR = DATA_DIR / 'cleaned'
OUTPUT_DIR.mkdir(exist_ok=True)

def read_csv_safe(path, **kwargs):
    try:
        return pd.read_csv(path, **kwargs)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding='latin-1', **kwargs)

df = read_csv_safe('employee_reviews.csv')
df.head()

,Unnamed: 0,company,location,dates,job-title,summary,pros,cons,advice-to-mgmt,overall-ratings,work-balance-stars,culture-values-stars,carrer-opportunities-stars,comp-benefit-stars,senior-mangemnet-stars,helpful-count,link
0,1,google,none,"Dec 11, 2018",Current Employee - Anonymous Employee,Best Company to work for,People are smart and friendly,Bureaucracy is slowing things down,none,5.0,4.0,5.0,5.0,4.0,5.0,0,https://www.glassdoor.com/Reviews/Google-Revie...
1,2,google,"Mountain View, CA","Jun 21, 2013",Former Employee - Program Manager,"Moving at the speed of light, burn out is inev...","1) Food, food, food. 15+ cafes on main campus ...",1) Work/life balance. What balance? All those ...,1) Don't dismiss emotional intelligence and ad...,4.0,2.0,3.0,3.0,5.0,3.0,2094,https://www.glassdoor.com/Reviews/Google-Revie...
2,3,google,"New York, NY","May 10, 2014",Current Employee - Software Engineer III,Great balance between big-company security and...,"* If you're a software engineer, you're among ...","* It *is* becoming larger, and with it comes g...",Keep the focus on the user. Everything else wi...,5.0,5.0,4.0,5.0,5.0,4.0,949,https://www.glassdoor.com/Reviews/Google-Revie...
3,4,google,"Mountain View, CA","Feb 8, 2015",Current Employee - Anonymous Employee,The best place I've worked and also the most d...,You can't find a more well-regarded company th...,I live in SF so the commute can take between 1...,Keep on NOT micromanaging - that is a huge ben...,5.0,2.0,5.0,5.0,4.0,5.0,498,https://www.glassdoor.com/Reviews/Google-Revie...
4,5,google,"Los Angeles, CA","Jul 19, 2018",Former Employee - Software Engineer,"Unique, one of a kind dream job",Google is a world of its own. At every other c...,"If you don't work in MTV (HQ), you will be giv...",Promote managers into management for their man...,5.0,5.0,5.0,5.0,5.0,5.0,49,https://www.glassdoor.com/Reviews/Google-Revie...


In [28]:
audit = pd.DataFrame({
    'column': df.columns,
    'dtype': df.dtypes.astype(str),
    'null_count': df.isna().sum(),
    'null_pct': (df.isna().sum() / len(df) * 100).round(2),
    'unique_count': [df[c].nunique(dropna=False) for c in df.columns]
})
audit.to_csv(OUTPUT_DIR / 'audit_employee_reviews.csv', index=False)
audit

,column,dtype,null_count,null_pct,unique_count
Unnamed: 0,Unnamed: 0,int64,0,0.00,67529
company,company,object,0,0.00,6
location,location,object,0,0.00,2044
dates,dates,object,1,0.00,3824
job-title,job-title,object,0,0.00,8308
summary,summary,object,127,0.19,42649
pros,pros,object,0,0.00,66085
cons,cons,object,0,0.00,66049
advice-to-mgmt,advice-to-mgmt,object,672,1.00,35190
overall-ratings,overall-ratings,float64,0,0.00,5


In [29]:
def clean_text(x):
    if pd.isna(x): return np.nan
    s = str(x).encode('ascii', 'ignore').decode().lower().strip()
    return re.sub(r'\s+', ' ', s)

# def parse_datetime_series(s, unit=None):
#     # tries numeric epoch or string datetime
#     s2 = pd.to_datetime(s, errors='coerce', unit=unit) if unit else pd.to_datetime(s, errors='coerce')
#     # if too many NaT and we didn't try unit, try 's' unit
#     if s2.isna().mean() > 0.9 and unit is None:
#         s2 = pd.to_datetime(s, errors='coerce', unit='s')
#     return s2

def clean_dates(col):
    s = col.astype(str).str.strip()
    dt = pd.to_datetime(s, errors="coerce")
    mask_excel = dt.isna() & s.str.fullmatch(r"\d+")
    if mask_excel.any():
        serials = s[mask_excel].astype(float)
        excel_dates = pd.to_datetime("1899-12-30") + pd.to_timedelta(serials, unit="D")
        dt[mask_excel] = excel_dates
    return dt


    # 3. If STILL NaT → try Excel serial conversion
    # Excel serial dates are > 1 and < 60000 (roughly)
    mask_excel = s2.isna() & s.astype(str).str.replace('.', '', 1).str.isnumeric()

    if mask_excel.sum() > 0:
        excel_vals = pd.to_numeric(s[mask_excel], errors="coerce")
        excel_dates = pd.to_datetime("1899-12-30") + pd.to_timedelta(excel_vals, unit="D")
        s2[mask_excel] = excel_dates

    return s2

def remove_outliers_3sigma(series):
    if series is None or series.isna().all(): return series
    x = series.astype(float); mu = x.mean(); sigma = x.std(ddof=0)
    if sigma == 0 or np.isnan(sigma): return series
    return series.where((x >= mu - 3*sigma) & (x <= mu + 3*sigma))

In [30]:
df.columns = [c.strip().lower().replace(' ', '_').replace('-', '_') for c in df.columns]
df = df.drop_duplicates()
text_cols = ['company', 'location', 'job_title', 'summary', 'pros', 'cons', 'advice_to_mgmt', 'link']
for col in text_cols:
    if col in df.columns: df[col] = df[col].apply(clean_text)

# if 'dates' in df.columns:
#     df['dates'] = pd.to_datetime(df['dates'], errors='coerce')
#     df['year'] = df['dates'].dt.year

if "dates" in df.columns:
    df["dates"] = clean_dates(df["dates"])
    df["year"] = df["dates"].dt.year
    df["dates"] = df["dates"].dt.strftime("%m-%d-%Y")


rating_cols = [c for c in df.columns if 'star' in c or 'rating' in c]
for col in rating_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = remove_outliers_3sigma(df[col])

if 'helpful_count' in df.columns:
    df['helpful_count'] = pd.to_numeric(df['helpful_count'], errors='coerce').fillna(0).astype(int)

df.drop(columns=['year'], errors='ignore', inplace=True)

# Rename the incorrect column name
df = df.rename(columns={"unnamed:_0": "id"})

# df['location'] = df['location'].replace('none', np.nan)
# df = df.dropna(subset=['company', 'job_title', 'summary'])
df.head()

,id,company,location,dates,job_title,summary,pros,cons,advice_to_mgmt,overall_ratings,work_balance_stars,culture_values_stars,carrer_opportunities_stars,comp_benefit_stars,senior_mangemnet_stars,helpful_count,link
0,1,google,none,12-11-2018,current employee - anonymous employee,best company to work for,people are smart and friendly,bureaucracy is slowing things down,none,5.0,4.0,5.0,5.0,4.0,5.0,0,https://www.glassdoor.com/reviews/google-revie...
1,2,google,"mountain view, ca",06-21-2013,former employee - program manager,"moving at the speed of light, burn out is inev...","1) food, food, food. 15+ cafes on main campus ...",1) work/life balance. what balance? all those ...,1) don't dismiss emotional intelligence and ad...,4.0,2.0,3.0,3.0,5.0,3.0,2094,https://www.glassdoor.com/reviews/google-revie...
2,3,google,"new york, ny",05-10-2014,current employee - software engineer iii,great balance between big-company security and...,"* if you're a software engineer, you're among ...","* it *is* becoming larger, and with it comes g...",keep the focus on the user. everything else wi...,5.0,5.0,4.0,5.0,5.0,4.0,949,https://www.glassdoor.com/reviews/google-revie...
3,4,google,"mountain view, ca",02-08-2015,current employee - anonymous employee,the best place i've worked and also the most d...,you can't find a more well-regarded company th...,i live in sf so the commute can take between 1...,keep on not micromanaging - that is a huge ben...,5.0,2.0,5.0,5.0,4.0,5.0,498,https://www.glassdoor.com/reviews/google-revie...
4,5,google,"los angeles, ca",07-19-2018,former employee - software engineer,"unique, one of a kind dream job",google is a world of its own. at every other c...,"if you don't work in mtv (hq), you will be giv...",promote managers into management for their man...,5.0,5.0,5.0,5.0,5.0,5.0,49,https://www.glassdoor.com/reviews/google-revie...


In [31]:
rating_cols = [
    "overall_ratings",
    "work_balance_stars",
    "culture_values_stars",
    "carrer_opportunities_stars",
    "comp_benefit_stars",
    "senior_mangemnet_stars"
]

# ---- CLEAN BLANKS ----
df[rating_cols] = df[rating_cols].replace(['', ' ', 'NA', 'N/A'], pd.NA)
df[rating_cols] = df[rating_cols].apply(pd.to_numeric, errors='coerce')

# ---- 1. IMPUTE USING (COMPANY, LOCATION) MEAN ----
group_means = df.groupby(["company", "location"])[rating_cols].transform('mean')
df[rating_cols] = df[rating_cols].fillna(group_means)

# ---- 2. IMPUTE USING COMPANY-ONLY MEAN ----
company_means = df.groupby("company")[rating_cols].transform('mean')
df[rating_cols] = df[rating_cols].fillna(company_means)

# ---- 3. IMPUTE USING GLOBAL MEAN AS FINAL FALLBACK ----
global_means = df[rating_cols].mean()
df[rating_cols] = df[rating_cols].fillna(global_means)

# ---- CONVERT TO INTEGER STARS ----
df[rating_cols] = df[rating_cols].round().astype("Int64")

# CHECK REMAINING MISSING VALUES
df[rating_cols].isna().sum()

overall_ratings               0
work_balance_stars            0
culture_values_stars          0
carrer_opportunities_stars    0
comp_benefit_stars            0
senior_mangemnet_stars        0
dtype: int64

In [32]:
df.describe()

,id,overall_ratings,work_balance_stars,culture_values_stars,carrer_opportunities_stars,comp_benefit_stars,senior_mangemnet_stars,helpful_count
count,67529.000000,67529.0,67529.0,67529.0,67529.0,67529.0,67529.0,67529.000000
mean,33765.000000,3.826075,3.378445,3.7507,3.665166,3.945224,3.323343,1.268211
std,19494.087501,1.154989,1.240152,1.185384,1.167802,0.983834,1.245872,16.085852
min,1.000000,1.0,1.0,1.0,1.0,1.0,1.0,0.000000
25%,16883.000000,3.0,3.0,3.0,3.0,3.0,3.0,0.000000
50%,33765.000000,4.0,4.0,4.0,4.0,4.0,3.0,0.000000
75%,50647.000000,5.0,4.0,5.0,5.0,5.0,4.0,1.000000
max,67529.000000,5.0,5.0,5.0,5.0,5.0,5.0,2094.000000


In [ ]:
# --- Load mapper with safe encoding ---
mapping = pd.read_csv(
    "company_location_mappings.csv",
    encoding="latin1"         # <-- handles é, ñ, ö, etc.
)

# Clean mapper columns we'll use
mapping["company_name"] = mapping["company_name"].astype(str).str.lower().str.strip()
mapping["company_id"] = mapping["company_id"].astype(str).str.lower().str.strip()

# --- Repeat the mapping rows until they cover all df rows ---
n_df = len(df)
n_map = len(mapping)

repeats = int(np.ceil(n_df / n_map))
mapping_repeated = (
    pd.concat([mapping] * repeats, ignore_index=True)
      .iloc[:n_df]                # trim to exact length of df
      .reset_index(drop=True)
)

# --- Overwrite company and *location → company_id* in reviews DF ---
df["company"]  = mapping_repeated["company_name"]
df["location"] = mapping_repeated["company_id"]

# Now make location become company_id (like your original code)
df.rename(columns={"location": "company_id"}, inplace=True)

# --- Preview ---
df.head(5)




,id,company,company_id,dates,job_title,summary,pros,cons,advice_to_mgmt,overall_ratings,work_balance_stars,culture_values_stars,carrer_opportunities_stars,comp_benefit_stars,senior_mangemnet_stars,helpful_count,link
0,1,ibm,1009,12-11-2018,current employee - anonymous employee,best company to work for,people are smart and friendly,bureaucracy is slowing things down,none,5,4,5,5,4,5,0,https://www.glassdoor.com/reviews/google-revie...
1,2,ge healthcare,1016,06-21-2013,former employee - program manager,"moving at the speed of light, burn out is inev...","1) food, food, food. 15+ cafes on main campus ...",1) work/life balance. what balance? all those ...,1) don't dismiss emotional intelligence and ad...,4,2,3,3,5,3,2094,https://www.glassdoor.com/reviews/google-revie...
2,3,hewlett packard enterprise,1025,05-10-2014,current employee - software engineer iii,great balance between big-company security and...,"* if you're a software engineer, you're among ...","* it *is* becoming larger, and with it comes g...",keep the focus on the user. everything else wi...,5,5,4,5,5,4,949,https://www.glassdoor.com/reviews/google-revie...
3,4,oracle,1028,02-08-2015,current employee - anonymous employee,the best place i've worked and also the most d...,you can't find a more well-regarded company th...,i live in sf so the commute can take between 1...,keep on not micromanaging - that is a huge ben...,5,2,5,5,4,5,498,https://www.glassdoor.com/reviews/google-revie...
4,5,accenture,1033,07-19-2018,former employee - software engineer,"unique, one of a kind dream job",google is a world of its own. at every other c...,"if you don't work in mtv (hq), you will be giv...",promote managers into management for their man...,5,5,5,5,5,5,49,https://www.glassdoor.com/reviews/google-revie...


In [34]:
DATE_COLUMNS = ['dates'] # Columns to fix
INPUT_FORMAT = '%m-%d-%Y'   # The format of your dates (e.g., "04-11-2024")
# --- End Configuration ---
try:

    for col in DATE_COLUMNS:
        if col in df.columns:
            print(f"Converting column: {col}...")
            # Convert text to datetime objects
            # errors='coerce' will turn any bad dates (like "ASAP") into NaT (NULL)
            temp_dates = pd.to_datetime(df[col], format=INPUT_FORMAT, errors='coerce')
            
            # Convert back to a string, but in the new YYYY-MM-DD format
            # NaT (NULL) values will be saved as empty strings
            df[col] = temp_dates.dt.strftime('%Y-%m-%d')
        else:
            print(f"Warning: Column '{col}' not found.")
except Exception as e:
    print(f"An error occurred: {e}")

Converting column: dates...


In [35]:
out = OUTPUT_DIR / 'employee_reviews_clean_with_impute.csv'
df.to_csv(out,date_format="%Y-%m-%d", index=False)
out

# out = OUTPUT_DIR / 'employee_reviews_clean_with_impute.xlsx'
# df.to_excel(out,date_format="%Y-%m-%d", index=False)
# out

WindowsPath('cleaned/employee_reviews_clean_with_impute.csv')